# Hard-wall endpoint alpha sweep — lane B

Nx=20, Ny=30, T=60, ten independent Born samples per alpha.
This lane runs [2, 1.9, 2.1, 1.7, 2.3, 1.2, 2.8], in that order. Run A/B/C simultaneously on **three separate A100s**;
never launch the same lane twice. Hard walls x=5,15; alpha2=30, nshell=1,
maximally mixed with canonical exterior preparation, slab-only raster-y,
perfect correction, complex128, no postselection. Allocator ceiling: 38 GiB.

Save complete endpoint spectra and one gap eigenmode per sample, no intermediate
entropy or spectra. Rates are lambda=-atanh(a)/60 and gap=min(abs(lambda)).
Centered caps within 1e-9 give infinite rates. Infinite gaps have a flagged display
placeholder 100; this is never a finite measurement or included in averages.
Mode vectors are zero padded with valid=False when all modes are capped.

One resident ten-sample batch per alpha; two durable five-sample shards.
Rolling covariance/RNG checkpoints every ten cycles, including cycle 60.
Relaunch this notebook to resume. Plan for 2–4 hours per lane, updated from measured timings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # optionally 1 for a first-batch check
LANE = 'B'  # fixed lane; do not change
# Full scientific configuration (changes require a new revision).
CONFIG = {'sampling_revision': 'hard_wall_alpha21_nx20_ny30_s10_t60_endpoint_modes_v1',
 'root_seed': 2026100131,
 'Nx': 20,
 'Ny': 30,
 'alpha_values': [1,
                  1.2,
                  1.4,
                  1.6,
                  1.7,
                  1.8,
                  1.85,
                  1.9,
                  1.95,
                  1.975,
                  2,
                  2.025,
                  2.05,
                  2.1,
                  2.15,
                  2.2,
                  2.3,
                  2.4,
                  2.6,
                  2.8,
                  3],
 'samples_per_case': 10,
 'cycles': 60,
 'execution_batch_size': 10,
 'result_shard_size': 5,
 'segment_cycles': 10,
 'DW': True,
 'dw_truncation': True,
 'meas_slab_only': True,
 'triv_region_local_mode': False,
 'alpha_2': 30.0,
 'nshell': 1,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'init_mode': 'maxmix',
 'n_a': 0.5,
 'sequence': 'raster_y',
 'perfect_correction': True,
 'postselect': False,
 'state_representation': 'covariance',
 'dtype': 'complex128',
 'device': 'cuda:0',
 'backend': 'local',
 'cap_tolerance': 1e-09,
 'initial_purity_tolerance': 0.50000001,
 'gpu_allocator_limit_bytes': 40802189312,
 'spectrum_cycles': [60],
 'observer_sample_chunk': 1,
 'covariance_spectral_clip': False,
 'gap_mode_vectors': True,
 'gap_placeholder': 100.0,
 'gap_tie_tolerance': 1e-12,
 'eigenmode_basis': 'active_centered_covariance_eigenvectors'}
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/31_hard_wall_alpha_endpoint_ny30')
LOCAL_BUNDLE_DIR = Path('/content/31_hard_wall_alpha_endpoint_ny30_lane_B')
SCRATCH_ROOT = Path('/content/alpha_endpoint_ny30_lane_B_scratch')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']


## Stage locally and run/resume
Live child stdout/stderr, cycle progress, checkpoint status and ETA. Each alpha has disjoint result/checkpoint paths.

In [ ]:
import codecs
import json
import shutil
import subprocess
import sys

assert LANE == 'B', 'Use the appropriate fixed-lane notebook'
LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'endpoint_spectrum.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py', 'analyze_campaign.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
           '--lane', LANE, '--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
           '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command.extend(['--max-new-execution-batches', str(MAX_NEW_EXECUTION_BATCHES)])
print('[launch]', ' '.join(command), flush=True)
decoder = codecs.getincrementaldecoder('utf-8')('replace')
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as process:
    try:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True)); sys.stdout.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print('[done] lane completed or requested batch limit reached', flush=True)


## Combined analysis
After all three lanes finish, run analyze_campaign.py locally against the downloaded output collection. It requires all 42 verified pairs and uses ordinary sample SEM, with infinite-gap placeholders excluded. No analysis writer runs concurrently here.

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
